# Notebook 2: Build a Simple LLM App with Gradio

![Workshop flow — you are here: Gradio Apps & Streaming](../assets/notebook_flow_diagram.png)

Notebook 1 was about prompts. This one is about **packaging** a prompt so someone else can use it without touching Python.

We will go in small steps: function first, interface second, streaming last.


## Step 1: Same setup as Notebook 1


In [1]:
import sys
from pathlib import Path

# Run this cell first. It finds the project folder and loads your .env file.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")


False

In [2]:
from src.llm_gateway import check_available_providers, run_llm, stream_llm

SELECTED_PROVIDER = "auto"
check_available_providers()


{'openai': False,
 'anthropic': False,
 'gemini': False,
 'mistral': False,
 'cohere': False,
 'deepseek': False,
 'groq': False,
 'openrouter': False,
 'ollama': True}

## Step 2: One function, one job

Before any buttons or sliders, we call the model from a plain function.


In [3]:
from src.gradio_apps import generate_learning_support
from src.output_formatting import show_llm_output

result = generate_learning_support(
    user_text="Transformers use attention to relate tokens in a sequence.",
    audience="undergraduate students",
    task_type="Explain simply",
    provider=SELECTED_PROVIDER,
)
show_llm_output(result, title="Learning support preview")


/Users/jeremiah/miniconda3/envs/llm/lib/python3.13/site-packages/gradio/routes.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Learning support preview

## Explain simply

# Explain Simply: Transformers Use Attention to Relate Tokens in a Sequence

## Short Answer: 2-3 sentences

Transformers are a type of neural network architecture used for tasks such as translation and language modeling. One key aspect of transformers is the use of attention mechanisms, which help the model focus on certain parts of the input sequence when processing a given token in the output sequence. This allows the model to better understand and generate coherent text.

## Key Points: Bullet List

* Transformers are a type of neural network architecture.
* They use attention mechanisms to relate tokens in a sequence.
* Attention helps the model focus on certain parts of the input sequence.
* This allows the model to better understand and generate coherent text.

## One Example: Concrete and Simple

Consider a simple example where we want to translate an English sentence into French. The transformer can use attention to focus on different parts of the English sentence when generating each token in the French translation. For example, if the English sentence is "I love dogs," the transformer might first pay attention to the word "love" when generating the French translation of "love." Then it would pay attention to the word "dogs" when generating the French translation of "dogs." This allows the model to produce a more accurate and coherent translation.

## One Common Misunderstanding: One Sentence

One common misunderstanding about transformers is that they only use attention in one direction. However, this is not the case. Transformers can use attention mechanisms in both directions, allowing them to process input sequences as well as output sequences. This makes transformers a very powerful and flexible tool for natural language processing tasks.

## Quiz: Two Short Questions

1. What is the main purpose of attention mechanisms in transformers?
* To help the model focus on certain parts of the input sequence when processing a given token in the output sequence.
1. Can transformers only use attention in one direction?
* No, transformers can use attention mechanisms in both directions.

If that worked, the hard part is done. The Gradio app below is mostly wiring.


## Step 3: Launch the learning support app

Uncomment `launch()` when you are ready. Stop the cell with the stop button when finished.


In [4]:
from src.gradio_apps import build_learning_support_app

app = build_learning_support_app()
app.launch()


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


Try the preset dropdown (Explain like a lecturer, Create study notes, etc.). Notice how the prompt template stays the same — only the settings change.


## Step 4: A simple chatbot


In [5]:
from src.gradio_apps import build_chatbot_app

chat = build_chatbot_app(streaming=False)
#chat.launch()

## Step 5: Streaming (why it feels faster)

Streaming prints tokens as they arrive. The total time is similar, but the experience feels quicker.


In [6]:
from IPython.display import Markdown, display

chunks = []
for chunk in stream_llm("Explain overfitting in one paragraph.", provider=SELECTED_PROVIDER):
    chunks.append(chunk)

display(Markdown("".join(chunks)))


 Overfitting occurs when a model or algorithm is trained on a dataset that contains too much noise or irrelevant information, resulting in the model becoming excessively complex and performing poorly on new, unseen data. This can happen when a model is trained with a large number of features or parameters relative to the size of the training data. In overfitting, the model may fit the training data very well but fail to generalize to new data, leading to poor performance in practice. To address overfitting, techniques such as regularization, cross-validation, and pruning can be used to simplify the model and improve its ability to generalize to new data. 

## Step 6: Streaming chatbot

In [7]:
stream_chat = build_chatbot_app(streaming=True)
stream_chat.launch()

Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


## Safety note

The apps warn users not to paste private data. In a real product you would also log errors, limit input length, and document what leaves the machine.

## Mini project ideas

Pick one: student support bot, lecture explainer, abstract helper, code error explainer, assignment feedback helper.

Add your own prompt template, provider choice, Gradio UI, and a short checklist for testing.
